# Process GESTALT barcodes from serially sampled sperm data 
**Authors:** <br>
Step A: Connor Shrader and Andy Sposato <br>
Step B: Connor Shrader and Andy Sposato <br>
Step C: Drew Shapiro and Connor Shrader <br>
Step D: Connor Shrader <br>
Step E: Connor Shrader and Andy Sposato <br>
Step F: Connor Shrader, Andy Sposato & Kimberly Truong <br>

### A) Throw out whole files that have low total reads.
Each file contains the barcode reads for a single fish, sample, and replicate. <br>
If the number of barcode reads in the file is less than a given threshold, the data is to unreliable to be used and is thrown out.

Data that pass this step are saved to `all_data.csv`.

### B) Filter individual barcodes by reads and proportion cutoffs. 
The data contains many barcodes that have very few reads. To reduce this noise, we define a threshold number of reads that a barcode must appear in a replicate to be kept. Each barcode must also exceed some specified *proportion* of the total reads in the replicate.

Data that pass this step are saved to `filtered_data.csv`.

### C) Cluster barcodes that are nearly identical
The data contain many barcodes that come from nearly identical DNA sequences. To distinguish barcode that only differ by misreads from truly distinct barcodes, we cluster the sequences in each fish. We measure "distance" between sequences using the Levenshtein distance metric and generate a network that connects nearby sequences. Clusters are chosen to be the connected components of this network.

Data after this step are saved to `collapsed_data.csv`.

### D) Additional filtering steps

Some additional filtering steps are run to remove barcodes that only appeared in one sample, or that never appear in all three replicates.

Data after this step are saved to `refined_data.csv`.

### E) Examine questionable barcodes and identify contaminant barcodes. 

GESTALT editing should produce unique barcodes within each fish. However, in the data, we sometimes see reads from the same barcode in multiple fish. This step identifies the fish to which the barcode truly belongs, and removes that barcode from other fish. When a source fish cannot be identified, the barcode is removed from all fish.

Data after this step are saved to `classified_data.csv`.

### F) Make barcode matrix. 
A composite barcode matrix file containing barcode information for all fish in the dataset is written to the experiments folder and used for generating Muller plots. Barcode matrix files for each individual fish are also written to the experiments folder.

## Warnings: 
This python notebook file should be stored at the same level as `data/`. <br>

This pipeline will break if you change your naming system for sample replicates or mess with the data structure Kimberly established. See github.com/Gagnon-lab/Sperm-Analysis for Kimberly's pipeline.

### Settings

Step A:
* `data_dir`: The data directory
* `dataset`: The subdirectory of `data/` that contains `.allReadCounts` files.
* `mergedinfos`: The subdirectory of `data/` that contains `merged.info` files.
* `total_read_cutoff`: ignore any `.allReadCounts` files with total reads strictly less than this value.
* `used_prop_cutoff`: ignore any `.allReadCounts` files with GESTALT read usage below this proportion.

In [1]:
data_dir = 'data'
dataset = 'full-barcoded-fish'
mergedinfos = 'mergedinfos'
total_read_cutoff = 1000
used_prop_cutoff = 0.3

Step B:
* `reads_cutoff`: remove a barcode from a sample if it has strictly less than this many reads.
* `prop_cutoff`: remove a barcode from a sample if it represents less than this proportion of the total reads.

In [2]:
reads_cutoff = 10
prop_cutoff = 0.005

Step C:
* `unedited`: The barcode of the unedited barcode.
* `cluster_barcodes`: If `False`, this step is not run.
* `edge_distance`: The maximum distance between two sequences to consider them connected. A larger value of `edge_distance` means more clustering.

In [3]:
unedited = "NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE_NONE"
cluster_barcodes = False
edge_distance = 1

Step D:
* `run_fish_triple_rep_filter`: remove a barcode from a fish if it never appears in all 3 replicates in any sample.
* `fish_one_sample_filter`: remove a barcode from a fish if it only appears in one time sample.

In [4]:
run_fish_triple_rep_filter = True
run_fish_one_sample_filter = True

Step E:
* `ratio_cutoff`: Any barcodes with a ratio score larger than this value are immediately assigned to the "winner" fish. Set this value to `False` to disable the ratio filter.
* `list_questionable_barcodes_before_classify`: Output a text file listing questionable barcodes before the classifying step.
* `list_questionable_barcodes_after_classify`: Output a text file listing barcodes that are *still* questionable after the classifying step.
* `image_questionable_barcodes_before_classify`: Plot a heatmap of all questionable barcodes before the classifying step.
* `image_questionable_barcodes_after_classify`: Plot a heatmap of all barcodes that are *still* questionable after the classifying step.
* `manual_classification`: a dictionary mapping barcodes to assigned outcomes. Ideally, this should be used only for debugging and testing. For example, setting `manual_classification = {'NONE_ABC_NONE': 'fish9'}` will assign fish9 as the "winner" fish. One can also set the value `'contaminant'` to remove the barcode.
* `delete_still_questionable_barcodes`: Whether to delete any barcodes that are still questionable after classification.

In [5]:
ratio_cutoff = 50

list_questionable_barcodes_before_classify = True
list_questionable_barcodes_after_classify = True
image_questionable_barcodes_before_classify = True
image_questionable_barcodes_after_classify = True

manual_classification = {'barcode': 'fish6'}
delete_still_questionable_barcodes = True

detect_anomalies = True
final_kept_read_cutoff_rep = 0.3
final_kept_read_cutoff_fish = 0.5

Step (F):
* `edited_fish_only`: If `False`, barcode matrices of control fish will be created, too.
* `fillna`: Whether to fill in empty values with 0's in the output barcode matrices.
* `plot_all_mullers`: Whether to plot Muller plots of all fish after most pipeline steps. (Plots are created only after step B and later. Earlier plots contain thousands of barcodes and take a long time to run).
* `plot_all_heatmaps`: Whether to plot heatmaps of all fish after most pipeline steps. Heatmaps contain similar information as the Muller plots. However, they contain additional numerical information at the expense of visual appeal.
* `step_summaries`: Whether to output `reads_per_step.csv` and `barcodes_per_step.csv`, which summarize the total number of reads and barcodes after each pipeline step.

In [6]:
edited_fish_only = False
fillna = False

plot_all_mullers = True
plot_all_heatmaps = True
output_step_summaries = True

### Import packages and set up directory.

In [7]:
import datetime
import os 
# import re
# import operator
# import collections
import numpy as np
import pandas as pd
from natsort import natsorted 
import matplotlib.pyplot as plt
import warnings
# import seaborn as sns
from textwrap import wrap
# import Levenshtein
# import networkx as nx
import inspect

import pipeline_utility as pu
# import BarcodeRemap as br

Make sure that we are in the current working directory (`sperm-data-analysis/`)

In [8]:
cwd = os.getcwd()
if os.path.basename(cwd) != 'sperm-data-analysis':
    raise Exception(f'Make sure you are running Python in the sperm-data-analysis directory. \
You are currently running in {cwd}.')
else:
    print(f'We are in the current working directory: {cwd}{os.sep}')

We are in the current working directory: /Users/connor/Documents/github/sperm-data-analysis/


Create a unique experiments subfolder to contain all the outputs from running the pipeline.

In [9]:
date = datetime.datetime.today().strftime('%m.%d.%Y_%I.%M_%p')
experiments_subfolder = os.path.join('experiments', date, '')
                            
# if the experiments_subfolder doesn't already exist, create it
if not os.path.exists('./experiments'):
    os.mkdir('./experiments')
if not os.path.exists(experiments_subfolder): 
    os.mkdir(experiments_subfolder)

print(f'All outputs will be sent to {experiments_subfolder}.')

All outputs will be sent to experiments/01.11.2026_04.40_PM/.


Print the settings used to a text file titled `pipeline_settings.txt`.

In [10]:
summary = open(experiments_subfolder + "pipeline_settings.txt", "w")

def write_setting_line(file, var_name, buff):
    """
    Write a line of setting information to a file using the variable name as a string.

    'file': the file object we are writing to.
    'var_name': string name of the variable to log.
    'buff':     spacing buffer between name and value.
    """
    frame = inspect.currentframe().f_back
    context = {**frame.f_globals, **frame.f_locals}

    value = context.get(var_name, "<undefined>")
    file.write(f"\t{(var_name + ' ').ljust(buff, '.')} {value}\n")
        
summary.write(f"Settings used during the pipeline run [{date}].\n\n")

summary.write("Step (A): throw out files with low read counts.\n")

buff = 21
write_setting_line(summary, 'dataset', buff)
write_setting_line(summary, 'mergedinfos', buff)
write_setting_line(summary, 'total_read_cutoff', buff)
write_setting_line(summary, 'used_prop_cutoff', buff)

summary.write("\nStep (B): filter barcodes.\n")
write_setting_line(summary, 'reads_cutoff', buff)
write_setting_line(summary, 'prop_cutoff', buff)

buff = 20
summary.write("\nStep (C): cluster barcodes.\n")
write_setting_line(summary, 'unedited', buff)
write_setting_line(summary, 'cluster_barcodes', buff)
write_setting_line(summary, 'edge_distance', buff)

buff = 34
summary.write("\nStep (D): additional filtering.\n")
# write_setting_line(summary, 'run_complete_triple_rep_filter', buff)
write_setting_line(summary, 'run_fish_triple_rep_filter', buff)
write_setting_line(summary, 'run_fish_one_sample_filter', buff)

buff = 34
summary.write("\nStep (E): identify contaminant barcodes.\n")
write_setting_line(summary, 'ratio_cutoff', buff)
summary.write('\n')

buff = 50
write_setting_line(summary, 'list_questionable_barcodes_before_classify', buff)
write_setting_line(summary, 'list_questionable_barcodes_after_classify', buff)
write_setting_line(summary, 'image_questionable_barcodes_before_classify', buff)
write_setting_line(summary, 'image_questionable_barcodes_after_classify', buff)

run_manual_classification = isinstance(manual_classification, dict) and len(manual_classification) > 1
buff = 34
summary.write('\n')
write_setting_line(summary, 'run_manual_classification', buff)
summary.write('\n')
if run_manual_classification:
    summary.write(f"\tManual classifications were as follows:\n")
    for key, value in manual_classification.items():
        buff = 4 + max([len(key) for key in manual_classification.keys()])
        
        write_setting_line(summary, value, key, buff)
    summary.write("\n")

buff = 38
write_setting_line(summary, 'delete_still_questionable_barcodes', buff)

buff = 25
summary.write("\nStep (F): create barcode matrices.\n")
write_setting_line(summary, 'edited_fish_only', buff)
write_setting_line(summary, 'fillna', buff)
write_setting_line(summary, 'plot_all_mullers', buff)
write_setting_line(summary, 'plot_all_heatmaps', buff)
write_setting_line(summary, 'output_step_summaries', buff)
summary.close() 

## A) Throw out whole files that have low total reads.

Each file contains the barcode reads for a single fish, sample, and replicate. <br>
If the number of barcode reads in the file is less than a given threshold, the data is to unreliable to be used and is thrown out.

Data that pass this step are saved to `all_data.csv`.

In [11]:
# Get list of file paths leading to .allReadCounts files
# At the same time, save the corresponding path "tuples" to a list.
# This will be used later to detect "empty" .allReadCounts files
# and output the correct values to the step A summary file.

mergedinfo_paths = []
filepaths = []
for dirpath, dirnames, filenames in os.walk(f'{data_dir}/{dataset}'):
   for filename in filenames:
        if filename.endswith('.allReadCounts'):
            filepath = os.path.join(dirpath, filename)
            filepaths.append(filepath)

for dirpath, dirnames, filenames in os.walk(f'{data_dir}/{mergedinfos}'):
   for filename in filenames:          
        if filename.endswith('merged.info'):
            filepath = os.path.join(dirpath, filename)
            mergedinfo_paths.append(filepath)

filepaths = natsorted(filepaths)
all_file_tuples = [pu.tuple_from_file(fp) for fp in filepaths]

print(f"An example .allReadCounts filepath in your directory:\n{filepaths[0]}.")
print(f"An example file tuple: {all_file_tuples[0]}.")
print(f"\nIn total, {len(filepaths)} .allReadCounts files were found.")

An example .allReadCounts filepath in your directory:
data/full-barcoded-fish/fish1_s1_rep1/fish1_s1_rep1.allReadCounts.
An example file tuple: ('fish1', '1', '1').

In total, 1272 .allReadCounts files were found.


In [12]:
mergedinfo_list = []
for filename in mergedinfo_paths:
    df = pd.read_csv(filename, sep='\t')
    mergedinfo_list.append(df)

mergedinfo_df = pd.concat(mergedinfo_list, ignore_index=True)
# Print duplicate merged.info entries, then keep only first occurence
print('Fish with duplicate merged.info entries:')
print(mergedinfo_df[mergedinfo_df.duplicated(subset='sample')]['sample'])
mergedinfo_df.drop_duplicates(subset='sample', keep='first', inplace=True, ignore_index=False)
mergedinfo_df.set_index('sample', inplace=True)

Fish with duplicate merged.info entries:
Series([], Name: sample, dtype: object)


Load all of the reads from `.allReadCounts` files to a pandas DataFrame called `all_data`.

Each `.allReadCounts` file is also assigned to either `kept_groups` (if the read count exceeds the threshold), `removed_groups` (if it is not sufficiently large), or `empty_groups` (if the file is empty).

In [13]:
kept_groups = []
removed_groups = []
empty_groups = []
reads_per_group = {}
used_props = {}

df_list = []
for fish, sample, replicate in all_file_tuples:
    filename = f'{fish}_s{sample}_rep{replicate}'

    read_file = os.path.join(data_dir, dataset, filename, f"{filename}.allReadCounts")

    if 'checkpoint' in read_file:
        print(replicate)
    
    try:
        df = pd.read_csv(read_file, sep='\t', index_col='array')
    except Exception as e:
        print(f"Error reading {read_file}: {e}")
        continue

    if df.empty:
        empty_groups.append((fish, sample, replicate))
        reads_per_group[(fish, sample, replicate)] = 0
        continue
    
    reads = df['count'].sum()
    try:
        mergedinfo = mergedinfo_df.loc[filename]
        used_prop = reads / mergedinfo['totalEntries']
        
    except Exception as e:
        print(f'Could not find mergedinfo for {filename}.')
        used_prop = 0

    # Edit the data frame and append it to our list of data frames.
    df.rename(columns={'event': 'barcode', 'count': 'reads'}, inplace=True)
    df.index.name = None
    df.insert(1, 'fish', fish)
    df.insert(2, 'sample', sample)
    df.insert(3, 'replicate', replicate)
    df.insert(4, 'file', filename)

    reads_per_group[(fish, sample, replicate)] = reads
    used_props[(fish, sample, replicate)] = used_prop

    # Determine whether the file passes our initial filter.
    pass_total_read = (reads >= total_read_cutoff) 
    pass_used_prop = (used_prop > used_prop_cutoff)
    df.insert(7, 'pass_total_read', pass_total_read)
    try:
        df.insert(8, 'pass_used_prop', pass_used_prop)
        #print(filename)
    except Exception as e:
        print(mergedinfo)

    passed_all = pass_total_read & pass_used_prop
    if passed_all:
        kept_groups.append((fish, sample, replicate))
    else:
        removed_groups.append((fish, sample, replicate))

    df_list.append(df)

Could not find mergedinfo for fishwater_s4_rep2.


In [14]:
print(f"""\
Of the {len(filepaths)} total files, {len(kept_groups)} passed \
the initial filter.
      
Of the {len(removed_groups)} files that did not pass, {len(empty_groups)} \
were completely empty.""")

Of the 1272 total files, 1114 passed the initial filter.

Of the 158 files that did not pass, 0 were completely empty.


In [15]:
# Merge the list of data frames into a single data frame.
all_data = pd.concat(df_list, ignore_index = True)
all_data['sample'] = all_data['sample'].astype(int)
print(f'The number of entries in all_data is {len(all_data)}. The first five rows are below:')
all_data.head()

The number of entries in all_data is 1023042. The first five rows are below:


,barcode,fish,sample,replicate,file,reads,proportion,pass_total_read,pass_used_prop
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512,True,True
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813,True,True
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530,True,True
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537,True,True
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327,True,True


#### Create Step A outputs and save.

In [16]:
filename = os.path.join(experiments_subfolder, 'all_data.csv')
all_data.to_csv(filename, index = False)

In [17]:
# Write a `passed-samples-info.csv file`.
sample_info = []
for fish, sample, replicate in kept_groups:
    sample_id = f'{fish}_s{sample}_rep{replicate}'
    sample_info.append({'sample_id': sample_id, 'fish': fish[4:], 
                        'age': pu.sample_map[int(sample)], 'replicate': replicate, 
                        'condition': 'control' if 'ctrl' in fish else 'edited'})
    
passed_samples_info = pd.DataFrame(sample_info)
passed_samples_info.to_csv(experiments_subfolder + 'passed_samples_info.csv', index = False)
passed_samples_info.head()

,sample_id,fish,age,replicate,condition
0,fish1_s1_rep1,1,5,1,edited
1,fish1_s1_rep2,1,5,2,edited
2,fish1_s1_rep3,1,5,3,edited
3,fish1_s2_rep1,1,6,1,edited
4,fish1_s2_rep2,1,6,2,edited


In [18]:
all_groups = kept_groups + removed_groups

summary = open(experiments_subfolder + 'stepA_summary.txt', 'w')

summary.write(f"""\
Summary of step A: Throw out whole files that have low total reads.

Your data contains {len(all_groups)} .allReadCounts files.
If a file contained at least {total_read_cutoff} total reads across all
barcodes, then it has passed this filter step.\n
""")

if len(removed_groups) == 0: 
    summary.write('All of your files passed this step.\n')
elif len(removed_groups) > 0:
    summary.write(f'{len(removed_groups)} files failed this step:\n\n')

    for group in removed_groups: 
        fish, sample, replicate = group
        sample_id = f'{fish}_s{sample}_rep{replicate}'
        
        summary.write(
            f"{sample_id + ' failed.':<25}"
            f"{str(reads_per_group[group]).rjust(6)} total reads, "
            f"{(100 * used_props[group]):6.2f}% used read proportion.\n"
        )
            
summary.write(f'\n{len(kept_groups)} files passed the filter.\n\n')

summary.write("""\
The contents of read counts from all files was written in a new file titled 'all_data.csv.
The contents of read counts that passed were copied to a new file titled 'pass_data.csv'
in the same subfolder.

The passed-samples-info.csv was also written and saved.
It contains only samples that pass this step.
""")
summary.close() 

## B) Filter barcodes by reads and proportion cutoffs.

Throw away all data in sample 3! If we come up with other rules to clear out data, we will make a separate rule for it.

In [19]:
all_data = all_data[all_data['sample'] != 3].copy()

In [20]:
pass_data = all_data.query('pass_total_read and pass_used_prop') \
                    .drop(columns=['pass_total_read', 'pass_used_prop']).copy()

pass_data['reads_filter'] = pass_data['reads'] >= reads_cutoff
pass_data['prop_filter'] = pass_data['proportion'] >= prop_cutoff

In [21]:
filtered_data = pass_data.query('reads_filter and prop_filter') \
                .drop(columns=['reads_filter', 'prop_filter']).copy()

In [22]:
filtered_data

,barcode,fish,sample,replicate,file,reads,proportion
0,12D+34_12D+60_NONE_NONE_3D+143_84D+169_84D+169...,fish1,1,1,fish1_s1_rep1,57559,0.750512
1,1D+38_28D+41_NONE_NONE_NONE_NONE_54D+199_54D+1...,fish1,1,1,fish1_s1_rep1,10646,0.138813
2,7D+36_12D+60_NONE_1I+119+A&5D+121_19D+130_4S+1...,fish1,1,1,fish1_s1_rep1,1958,0.025530
3,1D+38_28D+41_NONE_18D+113_NONE_NONE_54D+199_54...,fish1,1,1,fish1_s1_rep1,1345,0.017537
4,1D+38_28D+41_NONE_NONE_NONE_NONE_NONE_NONE_NON...,fish1,1,1,fish1_s1_rep1,792,0.010327
...,...,...,...,...,...,...,...
1022930,87D+36_87D+36_87D+36_87D+36_NONE_1D+172_NONE_N...,fishwater,18,4,fishwater_s18_rep4,2440,0.432011
1022931,NONE_NONE_NONE_NONE_1D+146_81D+170_81D+170_81D...,fishwater,18,4,fishwater_s18_rep4,1335,0.236367
1022932,87D+36_87D+36_87D+36_87D+36_NONE_NONE_NONE_NON...,fishwater,18,4,fishwater_s18_rep4,909,0.160942
1022933,17I+38+GTACATCGAGTGTATGC_28D+65_28D+65_NONE_NO...,fishwater,18,4,fishwater_s18_rep4,428,0.075779


In [23]:
# Track how many reads/barcodes were removed for each sample
pass_aggregate = pass_data.groupby(by = ['fish', 'sample', 'replicate'])['reads'] \
                          .agg(['size', 'sum']) \
                          .rename(columns = {'size': 'pass_size', 'sum': 'pass_sum'})

filtered_aggregate = filtered_data.groupby(by = ['fish', 'sample', 'replicate'])['reads'] \
                                  .agg(['size', 'sum']) \
                                  .rename(columns = {'size': 'filtered_size', 'sum': 'filtered_sum'})

step_b_summary = pd.concat([pass_aggregate, filtered_aggregate], axis = 1)
step_b_summary.head()

pass_size  pass_sum  filtered_size  filtered_sum
fish  sample replicate                                                  
fish1 1      1                710     76693              6         72943
             2                451     40635              8         38979
             3                630     52249              6         49230
      2      1                761     73584              8         70087
             2                697     63886              8         60537

#### Create step B outputs and save

In [24]:
filename = os.path.join(experiments_subfolder, 'pass_data.csv')
pass_data.to_csv(filename, index = False)

In [25]:
summary = open(experiments_subfolder + 'stepB_summary.txt', 'w')
summary.write(f"""\
Summary of step B: Filter barcodes by reads and proportion cutoffs.

For each row in `passed_data.csv`, if a barcode had fewer than {reads_cutoff} \
reads or a proportion below {prop_cutoff}, it was removed from the file. \
A list of barcodes that were removed was written to a sample file that ends \
with 'stepB_removed_barcodes.txt'. This may be useful to verify that the changes \
from .passReadCounts to .filteredReadCounts were accurate.

Here is a recap of the filtering for each sample:
*************************************************\n\n""")

for group in step_b_summary.index: 
    fish, sample, replicate = group
    sample_id = f'{fish}_s{sample}_rep{replicate}'

    pass_read_counts = step_b_summary.loc[group, 'pass_sum']
    pass_count = step_b_summary.loc[group, 'pass_size']
    filtered_read_counts = step_b_summary.loc[group, 'filtered_sum']
    filtered_count = step_b_summary.loc[group, 'filtered_size']
    
    reads_removed = pass_read_counts - filtered_read_counts
    summary.write(f'{sample_id} had {pass_read_counts} reads in the `passed_data.csv` file. Filtering removed {reads_removed} reads.\n')
    summary.write(f'There are now {filtered_read_counts} reads kept in the `filtered_data.csv` file.\n\n')
    
    barcodes_removed = pass_count-filtered_count
    summary.write(f'{sample_id} had {pass_count} barcodes in the `passed_data.csv` file. Filtering removed {barcodes_removed} barcodes.\n')
    summary.write(f'There are now {filtered_count} barcodes kept in the `filtered_data.csv` file.\n\n')
    summary.write('*************************************************\n\n')
                  
summary.close()

## C) Barcode clustering

In [26]:
collapsed_data = filtered_data.copy()

reference_seq = pu.BarcodeRemap().zebrafish_reference_sequence
if cluster_barcodes:
    collapsed_data['sequence'] = collapsed_data['barcode'].apply(lambda x: pu.get_sequence(x, reference_seq) \
                                                          .upper())

    # grouped_data combines the reads of each barcode from different samples and replicates.
    grouped_data = collapsed_data.groupby(['fish', 'sequence', 'barcode'], as_index = False) \
                                ['reads'].sum() \
                                .sort_values('reads', ascending = False) \
                                .reset_index(drop=True)

In [27]:
all_output = None
if cluster_barcodes:
    all_output = []
    
    for fish in all_data['fish'].unique():
        fish_data = grouped_data.query('fish == @fish') \
                                .reset_index(drop=True)
        sequences = np.array(fish_data['sequence']) # all the sequences that appear in the fish

        all_output.append(f'\n***************************\n\n')
        all_output.append(f"There are {len(sequences)} unique sequences in {fish}.\n\n")
    
        distances = pu.compute_levenshtein_matrix(sequences)
        components = pu.find_connected_components(distances, threshold = edge_distance)

        singleton_components = []

        # Each component corresponds to a cluster.
        for i, component in enumerate(components):
            comp_barcodes = fish_data['barcode'].iloc[list(component)]
            comp_data = fish_data[fish_data['barcode'].isin(comp_barcodes)]

            if len(comp_data) > 1:
                all_output += pu.summarize_component(i, comp_data, distances)
                all_output.append('\n')
            else: singleton_components.append(comp_data)

            # Identify which barcode is the 'true' barcode. All other
            # barcodes in this cluster are mapped to this one.
            winner = pu.resolve_winner_barcode(comp_data, unedited)
    
            collapsed_data.loc[
                (collapsed_data['fish'] == fish) & 
                (collapsed_data['barcode'].isin(comp_barcodes)), 'barcode'
            ] = winner

            filtered_data.loc[
                (filtered_data['fish'] == fish) & 
                (filtered_data['barcode'].isin(comp_barcodes)), 'clustered_barcode'
            ] = winner

        all_output.append('The following barcodes were not a part of any cluster:\n')
        for comp_data in singleton_components:
            row = comp_data.iloc[0]
            all_output.append(f"{row['reads']:>6}\t {row['barcode']}.\n")

In [28]:
if cluster_barcodes:
    # Combine rows that share the same (reassigned) barcode
    collapsed_data = (
        collapsed_data.groupby(['barcode', 'fish', 'sample', 'replicate', 'file'], as_index=False)
                      .agg({'reads': 'sum', 'proportion': 'sum'})
                      .sort_values(['fish', 'sample', 'replicate'])
    )
    collapsed_data.drop(columns = 'sequence', inplace = True, errors = 'ignore')

#### Create step C outputs and save

In [29]:
filename = os.path.join(experiments_subfolder, 'filtered_data.csv')
filtered_data.to_csv(filename, index = False)

In [30]:
summary = open(experiments_subfolder + "stepC_summary.txt", 'w')
summary.write(f"""\
Summary of step C: Cluster barcodes that are nearly identical.
""")
if all_output is None:
    summary.write(f'(THIS STEP IS DISABLED)')
else:
    summary.write(f'Below is a summary of the clustering done to each fish:\n')
    summary.writelines(all_output)
                  
summary.close()

## D) Additional filtering steps

In [31]:
new_columns = ['fish_triple_rep_classification', 'fish_one_sample_classification']

for new_column in new_columns:
    collapsed_data.loc[:, new_column] = True
    collapsed_data[new_column] = collapsed_data[new_column].astype('bool')

Determine values of `fish_triple_rep_classification`, and `fish_one_sample_classification`.

In [32]:
barcodes = collapsed_data['barcode'].unique()

for barcode in barcodes:
    single_barcode = collapsed_data.query('barcode == @barcode')
    
    # grouped is a DataFrame with columns (fish, sample, replicate). The value of 'replicate' is
    # the count of replicates in which there is at least one read for that fish and sample.
    grouped = single_barcode.groupby(['fish', 'sample'], observed=True, as_index=False) \
                            ['replicate'].count() \
                            .reset_index()

    # if grouped.replicate.max() < 3 and run_complete_triple_rep_filter:
    #     collapsed_data.loc[
    #         collapsed_data['barcode'] == barcode, 
    #         'complete_triple_rep_classification'
    #     ] = False
        
    for fish in single_barcode['fish'].unique():
        # grouped is a DataFrame with columns (sample, replicate). The value of 'replicate' is the count
        # of replicates in which there is at least one read for that sample.grouped_fish = single_barcode.query("fish == @fish").groupby('sample')['replicate'].count().reset_index()
        grouped_fish = single_barcode.query('fish == @fish').groupby('sample') \
                                     ['replicate'].count() \
                                     .reset_index()

        if grouped_fish.replicate.max() < 3 and run_fish_triple_rep_filter:
            # never appears in 3 reps, throw out
            collapsed_data.loc[
                (collapsed_data['barcode'] == barcode) & (collapsed_data['fish'] == fish), 
                'fish_triple_rep_classification'
            ] = False
            
        if len(grouped_fish) == 1 and run_fish_one_sample_filter:
            # only appears in one sample, throw out
            collapsed_data.loc[
                (collapsed_data['barcode'] == barcode) & (collapsed_data['fish'] == fish), 
                'fish_one_sample_classification'
            ] = False

In [33]:
# refined_data contains all the rows of collapsed_data such that the three classification
# columns (defined above) are `True`.
refined_data = collapsed_data[collapsed_data.filter(like='_classification').all(True)].copy()
refined_data.drop(columns=[
    # 'complete_triple_rep_classification',
    'fish_triple_rep_classification',
    'fish_one_sample_classification'], inplace = True)

#### Create step D outputs and save

In [34]:
filename = os.path.join(experiments_subfolder, 'collapsed_data.csv')
collapsed_data.to_csv(filename, index = False)

In [35]:
refine_summary = collapsed_data.groupby(['fish', 'barcode'], observed=False).agg({
    'fish_triple_rep_classification': 'first',
    'fish_one_sample_classification': 'first'
}).reset_index()

In [36]:
summary = open(experiments_subfolder + "stepD_summary.txt", 'w')
summary.write(f"""\
Summary of step D: Additional filtering

Below is a summary of the additional barcodes that were filtered out from each fish.
""")
for fish in refine_summary['fish'].unique():
    refine_fish = refine_summary.query('fish == @fish')

    summary.write(f"\n=== Fish: {fish} ===\n")

    # Barcodes passing both filters
    passed = refine_fish[
        (refine_fish['fish_triple_rep_classification']) &
        (refine_fish['fish_one_sample_classification'])
    ]['barcode'].tolist()
    summary.write("\nBarcodes passing both filters:\n")
    for b in passed:
        summary.write(f"  {b}\n")

    # Barcodes failing triple_rep filter
    failed_triple = refine_fish[
        ~refine_fish['fish_triple_rep_classification']
    ]['barcode'].tolist()
    summary.write("\nBarcodes failing triple_rep filter:\n")
    for b in failed_triple:
        summary.write(f"  {b}\n")

    # Barcodes failing one_sample filter
    failed_one_sample = refine_fish[
        ~refine_fish['fish_one_sample_classification']
    ]['barcode'].tolist()
    summary.write("\nBarcodes failing one_sample filter:\n")
    for b in failed_one_sample:
        summary.write(f"  {b}\n")

    summary.write('\n*************************************************\n\n')
                  
summary.close()

## E) Examine questionable barcodes and identify contaminant barcodes. 

Questionable barcodes are barcodes that show up in multiple animals. They are likely contamination. We want to confidently remove them from samples where they do not belong. Before we can remove them from the appropriate files, we need to examine each according to the number of reads that support them in every fish. 

Find questionable barcodes, barcodes that show up in multiple fish. 

A barcode will not be tagged questionable if it shows up in multiple samples or replicates of a single fish. 

In [37]:
questionable_barcodes = pu.identify_questionable(refined_data, unedited)
print(f'There are {len(questionable_barcodes)} questionable barcodes.')

There are 26 questionable barcodes.


In [38]:
refined_data.loc[:, 'manual_classification'] = True
refined_data.loc[:, 'ratio_classification'] = True

In [39]:
for key, value in manual_classification.items():
    if value == 'contaminant':
        if (refined_data['barcode'] == key).any():
            refined_data.loc[refined_data['barcode'] == key, 'manual_classification'] = False
        else:
            warnings.warn(f'Manually assigned contaminant barcode {key} does not appear in refined_data.')
    else:
        if ((refined_data['barcode'] == key) & (refined_data['fish'] != value)).any():
            refined_data.loc[
                (refined_data['barcode'] == key) & (refined_data['fish'] != value),
                'manual_classification'
            ] = False
        else:
            warnings.warn(f"Manually assigned barcode '{key}' does not appear outside of {value}.")

/var/folders/59/35nb2tq50vz3vpwjkwkzppvh0000gn/T/ipykernel_18043/1387157648.py:14: UserWarning: Manually assigned barcode 'barcode' does not appear outside of fish6.
  warnings.warn(f"Manually assigned barcode '{key}' does not appear outside of {value}.")


In [40]:
questionable_after_manual = pu.identify_questionable(
    refined_data[refined_data['manual_classification']], unedited
)

if manual_classification != {}:
    print(f'Before manual classification, {len(questionable_barcodes)} barcodes were questionable. ' +
          f'{len(manual_classification)} had manual assignment. {len(questionable_after_manual)} ' +
          f'are still questionable.')

Before manual classification, 26 barcodes were questionable. 1 had manual assignment. 26 are still questionable.


In [41]:
if ratio_cutoff is not False:
    for barcode in questionable_after_manual:
        aggregate, ratio = pu.summarize_barcode(refined_data, barcode)
        winner_fish = aggregate.fish.iloc[0]

        if ratio > ratio_cutoff:
            refined_data.update(
                refined_data.query('barcode == @barcode and fish != @winner_fish') \
                            .assign(ratio_classification = False)
            )

In [42]:
classified_data = refined_data.query("manual_classification and ratio_classification").copy()
classified_data.drop(columns=[
    'manual_classification',
    'ratio_classification'], inplace = True)

Remove any still questionable barcodes if `delete_still_questionable_barcodes` is True.

#### Create step E outputs and save

In [43]:
filename = os.path.join(experiments_subfolder, 'refined_data.csv')
refined_data.to_csv(filename, index = False)

In [44]:
questionable_after_ratio = pu.identify_questionable(classified_data, unedited)
# classified_data['no_longer_questionable'] = ~classified_data['barcode'].isin(questionable_after_ratio)
print(f'There are {len(questionable_after_ratio)} barcodes that are still questionable.')

There are 20 barcodes that are still questionable.


In [45]:
questionable_barcodes_summary = dict()
for barcode in questionable_barcodes:
    query = classified_data.query('barcode == @barcode')['fish'].unique()
    
    if barcode in manual_classification:
        questionable_barcodes_summary[barcode] = f'{manual_classification[bc]} (manual classification)'
    elif barcode in questionable_after_ratio:
        questionable_barcodes_summary[barcode] = "still questionable"
    elif len(query) == 0:
        questionable_barcodes_summary[barcode] = "contaminant"
    elif len(query) == 1:
        fish = query[0]
        questionable_barcodes_summary[barcode] = f'{fish} (found by ratio)'
    else:
        warnings.warn(f"Something is wrong with barcode '{barcode}'.")

# write questionable barcodes to a file
if list_questionable_barcodes_before_classify:
    filename = os.path.join(experiments_subfolder, 'questionable_barcodes_list.txt')
    output = open(filename, 'w')
    for barcode in questionable_barcodes: 
        output.write(f'{barcode}: {questionable_barcodes_summary[barcode]}')
        output.write("\n")
    output.close()

if list_questionable_barcodes_after_classify and len(questionable_after_ratio) > 0:
    filename = os.path.join(experiments_subfolder, 'still_questionable_barcodes_list.txt')
    output = open(filename, 'w')
    for barcode in questionable_after_ratio: 
        output.write(barcode)
        output.write("\n")
    output.close()

In [46]:
# produce heatmaps
if image_questionable_barcodes_before_classify:
    directory = os.path.join(experiments_subfolder, 'heatmaps_before_classification')
    if not os.path.exists(directory): 
        os.mkdir(directory)
    data = refined_data.copy()
    for i, bc in enumerate(questionable_barcodes):
        _, ratio = pu.summarize_barcode(data, bc)
        fig, ax = pu.heatmap_barcode(data, bc)
        ax.set_title("\n".join(wrap(f"{bc}", 50) + [f"\nratio: {ratio:.2f}, outcome: {questionable_barcodes_summary[bc]}"]))
        fig.tight_layout()
        plt.savefig(os.path.join(directory, f"barcode[{i}].png"))
        plt.close()

if image_questionable_barcodes_after_classify:
    directory = os.path.join(experiments_subfolder, 'heatmaps_after_classification')
    if not os.path.exists(directory): 
        os.mkdir(directory)
    data = classified_data.copy()
    for i, bc in enumerate(questionable_barcodes):
        if bc not in questionable_after_ratio: continue
        _, ratio = pu.summarize_barcode(data, bc)
        fig, ax = pu.heatmap_barcode(data, bc)
        ax.set_title("\n".join(wrap(f"{bc}", 50) + [f"\nratio: {ratio:.2f}, outcome: {questionable_barcodes_summary[bc]}"]))
        fig.tight_layout()
        plt.savefig(os.path.join(directory, f"barcode[{i}].png"))
        plt.close()

In [47]:
if delete_still_questionable_barcodes:
    classified_data = classified_data[~classified_data['barcode'].isin(questionable_after_ratio)].copy()

# classified_data.drop(columns = 'no_longer_questionable', inplace = True)

In [48]:
filename = os.path.join(experiments_subfolder, 'classified_data.csv')
classified_data.to_csv(filename, index = False)

In [49]:
summary = open(experiments_subfolder+"stepE_summary.txt", 'w')
summary.write(
"""
Summary of step E: Remove contaminant barcodes from their read files.

The barcodes in questionable barcodes list were examined carefully
for their presence across animals. If a clear source of that barcode
could be found, the questionable barcode was assigned a source fish.
If there was not a clear winner, the questionable barcode was
considered contaminant everywhere.

The data before contaminant detection is stored in `refined_data.csv`.
The data after contaminant detection is stored in `classified_data.csv`.
Data with the contaminants removed is stored in `final_data.csv`.

Here is a recap of the filtering for each sample:
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
""")

for fish in refined_data['fish'].unique():
    fish_barcodes = refined_data.query('fish == @fish')['barcode'].unique()
    summary.write(f'\n === FISH {fish} ===\n\n')
    
    for barcode in fish_barcodes: 
        if barcode in questionable_barcodes:
            outcome = questionable_barcodes_summary[barcode]
            summary.write(f'{barcode}: {outcome}\n')

            if outcome == 'still questionable':
                fish_found = refined_data.query('barcode == @barcode')['fish'].unique()
                summary.write(f'This barcode is found in {fish_found}.\n')

            summary.write('\n')

summary.close()

## F) Make barcode matrix & final outputs.

In [50]:
# For each fish, sample, replicate, count the proportion of reads in the final
# dataset compared to the allReadCounts data.
prop_reads_kept = classified_data.groupby(['fish', 'sample', 'replicate', 'file'], as_index = False)['proportion'].sum()
prop_reads_kept_fish = prop_reads_kept.groupby(['fish'], as_index = False)['proportion'].mean()

In [51]:
# ADDITIONAL FILTERING
# Filter out files with less than a proportion of 0.3 kept reads
# Filter out fish with less than a proportion of 0.5 kept reads
# Code written by Drew Shapiro
prop_reads_kept['pass_prop'] = prop_reads_kept['proportion'] > final_kept_read_cutoff_rep
prop_reads_kept_fish['pass'] = prop_reads_kept_fish['proportion'] > final_kept_read_cutoff_fish

means = {}
for fish in prop_reads_kept['fish'].unique():
    filtered_prop_series = prop_reads_kept[(prop_reads_kept['fish'] == fish) & (prop_reads_kept['pass_prop'])]['proportion']
    means[fish] = (filtered_prop_series.mean(), filtered_prop_series.std())

prop_reads_kept['mean'] = prop_reads_kept['fish'].apply(lambda fish: means[fish][0])
prop_reads_kept['std'] = prop_reads_kept['fish'].apply(lambda fish: means[fish][1])
prop_reads_kept['pass_z'] = ~(prop_reads_kept['proportion'] < prop_reads_kept['mean'] - 3 * prop_reads_kept['std'])
# print(prop_reads_kept['pass'].value_counts())

# prop_reads_kept_fish = prop_reads_kept_fish[prop_reads_kept_fish['pass']]['fish']
if detect_anomalies:
    passing_files = prop_reads_kept[(prop_reads_kept['pass_z']) & (prop_reads_kept['pass_prop'])]['file']
else:
    passing_files = prop_reads_kept[(prop_reads_kept['pass_prop'])]['file']
passing_fish = prop_reads_kept_fish[prop_reads_kept_fish['pass']]['fish']

final_data = classified_data[(classified_data['file'].isin(passing_files)) & (classified_data['fish'].isin(passing_fish))].copy()

prop_reads_kept.to_csv(experiments_subfolder + 'prop_reads_kept.csv', index = False)
prop_reads_kept.to_csv(experiments_subfolder + 'prop_reads_kept_fish.csv', index = False)
prop_reads_kept.head()

,fish,sample,replicate,file,proportion,pass_prop,mean,std,pass_z
0,fish1,1,1,fish1_s1_rep1,0.951104,True,0.909801,0.054992,True
1,fish1,1,2,fish1_s1_rep2,0.959247,True,0.909801,0.054992,True
2,fish1,1,3,fish1_s1_rep3,0.942219,True,0.909801,0.054992,True
3,fish1,2,1,fish1_s2_rep1,0.952476,True,0.909801,0.054992,True
4,fish1,2,2,fish1_s2_rep2,0.947578,True,0.909801,0.054992,True


In [52]:
# Convert proportions so that they sum to 1 in each fish, sample, replicate
# NOTE: I commented this out so I can track how much is lost in each replicate. -Connor

# final_data.loc[:, 'proportion'] = (
#     final_data['proportion'] /
#     final_data.groupby(['fish', 'sample', 'replicate'])['proportion'].transform('sum')
# )

In [53]:
filename = os.path.join(experiments_subfolder, 'final_data.csv')
final_data.to_csv(filename, index = False)

In [54]:
# Create barcode matrices.
d = final_data.copy()
d['sample'] = pd.to_numeric(d['sample'])
d['sample_id'] = d['fish'] + '_s' + d['sample'].apply(str) + '_rep' + d['replicate']

directory = experiments_subfolder + 'matrix'
if not os.path.exists(directory): 
    os.mkdir(directory)

fish_list = d.fish.unique()
for fish in fish_list:
    fish_df = d[d['fish'] == fish].copy()
    pt = fish_df.pivot_table(columns = ['sample_id'], index = 'barcode', values = 'reads', sort = False)
    if fillna:
        pt.fillna(0, inplace = True)

    filename = os.path.join(experiments_subfolder, 'matrix', f'{fish}_barcodeMatrix_3filters.tsv')
    pt.to_csv(filename, sep = '\t')

pt = d.pivot_table(columns = ['sample_id'], index = 'barcode', values = 'reads', sort = False)
if fillna:
    pt.fillna(0, inplace = True)
    
filename = os.path.join(experiments_subfolder, 'matrix', f'barcodeMatrix_3filters.tsv')
pt.to_csv(filename, sep = '\t')

In [55]:
final_sequences = final_data[['fish', 'barcode']].drop_duplicates().reset_index()

In [56]:
# Print barcode, fish, and sequence into a new output file.
final_sequences = final_data[['fish', 'barcode']].drop_duplicates().reset_index(drop=True)
final_sequences['sequence'] = final_sequences['barcode'].apply(lambda x: pu.get_sequence(x, reference_seq).upper())

filename = os.path.join(experiments_subfolder, f'final_sequences.csv')
final_sequences.to_csv(filename)

In [57]:
# Truncate to first four edit edits

In [58]:
def subset_before_fourth_underscore(s):
    parts = s.split("_")
    if len(parts) <= 4:
        # fewer than 4 underscores → return original string
        return s
    return "_".join(parts[:4])

print(subset_before_fourth_underscore(unedited))

NONE_NONE_NONE_NONE


In [59]:
final_data_trunc = final_data.copy()
final_data_trunc['barcode'] = final_data_trunc['barcode'].apply(subset_before_fourth_underscore)

In [60]:
final_data_trunc = final_data_trunc.groupby(['fish', 'sample', 'replicate', 'barcode'], as_index=False).agg({'reads': 'sum', 'proportion': 'sum'})

filename = os.path.join(experiments_subfolder, 'final_data_trunc.csv')
final_data_trunc.to_csv(filename, index = False)

In [61]:
summary = open(experiments_subfolder+"stepF_summary.txt", 'w')
summary.write("Summary of step E: A barcode matrix file is written to the experiments folder and\nis used for generating Muller plots.\n\n")
summary.write("A barcode matrix file is also written for each fish in your samples.\n")
summary.write("The barcodes present in these matrix files will have passed all three filters (steps A, B and D)")
summary.write("\n")
summary.close()

In [62]:
datasets = ['all_data', 'pass_data', 'filtered_data', 'collapsed_data', 'refined_data', 'classified_data', 'final_data', 'final_data_trunc']

if output_step_summaries:
    fish_read_dicts = []
    fish_bc_dicts = []
    for name in datasets:
        dataset = globals()[name]
    
        # Group once instead of querying repeatedly
        grouped = dataset.groupby('fish')
    
        # Sum reads per fish
        fish_read_dict = grouped['reads'].sum().to_dict()
    
        # Count unique barcodes per fish
        fish_bc_dict = grouped['barcode'].nunique().to_dict()
    
        fish_read_dicts.append(fish_read_dict)
        fish_bc_dicts.append(fish_bc_dict)

    reads_df = pd.DataFrame(fish_read_dicts)
    bc_df = pd.DataFrame(fish_bc_dicts)

    reads_df = reads_df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    reads_df.index = pd.Index(datasets)
    reads_df = reads_df.T
    reads_df.loc['total'] = reads_df.sum(numeric_only=True)
    
    bc_df = bc_df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    bc_df.index = pd.Index(datasets)
    bc_df = bc_df.T
    bc_df.loc['total'] = bc_df.sum(numeric_only=True)

    filename = os.path.join(experiments_subfolder, 'reads_per_step.csv')
    reads_df.to_csv(f'{experiments_subfolder}reads_per_step.csv')
    
    filename = os.path.join(experiments_subfolder, 'barcodes_per_step.csv')
    bc_df.to_csv(f'{experiments_subfolder}barcodes_per_step.csv')

In [63]:
import importlib
importlib.reload(pu)
import pipeline_utility as pu

In [64]:
path = f'{experiments_subfolder}muller'
if not os.path.exists(path):
    os.mkdir(path)

for name in datasets[2:]:
    dataset = globals()[name]
    fishes = dataset['fish'].unique()

    path = f'{experiments_subfolder}muller/{name}'
    if not os.path.exists(path):
        os.mkdir(path)
    for fish in fishes:
        fig, ax = plt.subplots()
        pu.plot_muller(dataset, fish, ax = ax, add_legend = True)

        if plot_all_mullers:
            plt.savefig(f'{path}/{fish}_muller.png', bbox_inches = 'tight');
        plt.close()

        fig, ax = pu.heatmap_fish(dataset, fish)

        if plot_all_heatmaps:
            plt.savefig(f'{path}/{fish}_heatmap.png', bbox_inches = 'tight')
        plt.close()

/Users/connor/Documents/github/sperm-data-analysis/pipeline_utility.py:318: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(ts[0], ts[-1])
/Users/connor/Documents/github/sperm-data-analysis/pipeline_utility.py:318: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(ts[0], ts[-1])
/Users/connor/Documents/github/sperm-data-analysis/pipeline_utility.py:318: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(ts[0], ts[-1])
/Users/connor/Documents/github/sperm-data-analysis/pipeline_utility.py:318: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.set_xlim(ts[0], ts[-1])
/Users/connor/Documents/github/sperm-data-analysis/pipeline_utility.py:318: UserWarning: Attempting to set identical

# Job report

In [65]:
summary = open(experiments_subfolder+"full_job_report.txt", 'w')
stepA = open(experiments_subfolder+"stepA_summary.txt", 'r')
stepB = open(experiments_subfolder+"stepB_summary.txt", 'r')
stepC = open(experiments_subfolder+"stepC_summary.txt", 'r')
stepD = open(experiments_subfolder+"stepD_summary.txt", 'r')
stepE = open(experiments_subfolder+"stepE_summary.txt", 'r')

summary.write("Processed GESTALT barcodes from serially sampled sperm data.\n")
summary.write("Your job completed at " + datetime.datetime.now().strftime('%I:%M %p on %m/%d/%Y'))
summary.write("\nAndy Sposato wrote original code for steps A-D.\nAndy adapted the code in Step E from original code written by Kimberly Truong.")
summary.write("\n\n############################################################################################\n\n")
for line in stepA.readlines(): 
    summary.write(line)
stepA.close()
summary.write("\n############################################################################################\n\n")
for line in stepB.readlines(): 
    summary.write(line)
stepB.close()
summary.write("############################################################################################\n\n")
for line in stepC.readlines(): 
    summary.write(line)
stepC.close()
summary.write("############################################################################################\n\n")
for line in stepD.readlines(): 
    summary.write(line)
stepD.close()
summary.write("############################################################################################\n\n")
for line in stepE.readlines(): 
    summary.write(line)
stepE.close()
summary.write("\n############################################################################################\n\n")
summary.write("It was a pleasure filtering your sperm barcodes. \n")
summary.close()